# Day 099 | Kaggle Competition Submission

## 100 Days of ML for Geotechnical Engineering
**Phase:** Geotechnical AI Projects | **Author:** Ripon Chandra Malo | **Date:** 2026-05-16

---

## Geotechnical Problem

**Liquefaction susceptibility** is a life-safety problem. Saturated, loose, sandy soils lose shear strength during earthquake shaking — causing ground to flow like a liquid. The 2010–2011 Canterbury earthquakes in New Zealand caused over NZD 15 billion in liquefaction damage.

**Competition framing:** Given borehole SPT data, water table depth, and seismic parameters, predict the **probability** that a site will liquefy during a design-level earthquake event.

### Dataset Features

| Feature | Description | Units |
|---------|------------|-------|
| `depth` | Depth of critical layer | m |
| `N_value` | SPT N60 (corrected) | blows/300mm |
| `fines_content` | % passing 0.075 mm sieve | % |
| `water_table` | Depth to groundwater | m |
| `PGA` | Peak Ground Acceleration | g |
| `magnitude` | Earthquake moment magnitude | Mw |
| `Vs30` | Shear wave velocity, top 30 m | m/s |
| `liquefied` | **Target (train only):** 1=liquefied, 0=no | — |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTS & SETTINGS
# ─────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    HAS_XGB = True
    print(f'XGBoost {xgb.__version__} available.')
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print('XGBoost not installed — using sklearn GradientBoostingClassifier as fallback.')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---

## Part 1 — Kaggle Structure Explained

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# KAGGLE COMPETITION ANATOMY
# ─────────────────────────────────────────────────────────────────────────
structure = """
Kaggle Competition Structure — Liquefaction Prediction Challenge
═══════════════════════════════════════════════════════════════

Files provided:
  train.csv        ← 200 rows  | features + liquefied (label)
  test.csv         ←  50 rows  | features ONLY (no label)
  sample_submission.csv  ← id, liquefaction_probability=0.5 (placeholder)

Your task:
  1. Learn from train.csv (you can see the ground truth)
  2. Predict on test.csv  (no ground truth — Kaggle holds it)
  3. Create submission.csv  columns: id, liquefaction_probability
  4. Upload → Kaggle scores against the hidden ground truth

Metric: AUC-ROC
  - Best for imbalanced binary problems (only ~25% liquefied)
  - Random baseline: 0.500
  - Good model:      0.850+
  - State-of-the-art ~0.950

Two leaderboards:
  Public LB  ← scored on 20% of test set (updated every submission)
  Private LB ← scored on 80% of test set (revealed at competition end)
  Danger: overfitting to Public LB → Private LB drop ('shakeup')
"""
print(structure)

---

## Part 2 — Generate Mock Datasets

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GENERATE REALISTIC SYNTHETIC TRAIN AND TEST SETS
# ─────────────────────────────────────────────────────────────────────────
np.random.seed(42)

def generate_liquefaction_dataset(n: int, include_label: bool = True,
                                   id_offset: int = 0) -> pd.DataFrame:
    """Generate synthetic SPT-based liquefaction dataset.

    Features are sampled from realistic geotechnical distributions.
    Label is deterministic function of key parameters for consistency.

    Args:
        n: Number of records.
        include_label: Whether to add 'liquefied' column.
        id_offset: Starting row ID.

    Returns:
        DataFrame with id + features (+ label if include_label).
    """
    df = pd.DataFrame({
        'id':           range(id_offset, id_offset + n),
        'depth':        np.round(np.random.uniform(1.5, 12.0, n), 2),
        'N_value':      np.random.randint(4, 36, n).astype(float),
        'fines_content':np.round(np.random.uniform(0.0, 40.0, n), 1),
        'water_table':  np.round(np.random.uniform(0.5, 6.0, n), 2),
        'PGA':          np.round(np.random.uniform(0.05, 0.65, n), 3),
        'magnitude':    np.round(np.random.uniform(5.9, 8.2, n), 1),
        'Vs30':         np.round(np.random.uniform(100, 420, n), 0),
    })

    if include_label:
        # Geophysically motivated: liquefy when N_value low, PGA high,
        # depth below WT, and Vs30 low. Add noise.
        score = (
            -0.15 * df['N_value']
            + 4.0  * df['PGA']
            + 0.08 * (df['depth'] - df['water_table'])
            + 0.008 * df['fines_content']
            - 0.004 * df['Vs30']
            + 0.25 * df['magnitude']
            + np.random.normal(0, 0.4, n)
        )
        prob = 1 / (1 + np.exp(-score))
        df['liquefied'] = (prob > 0.50).astype(int)

    return df


train_df = generate_liquefaction_dataset(200, include_label=True, id_offset=0)
test_df  = generate_liquefaction_dataset(50,  include_label=False, id_offset=200)

# Save to disk (mimicking Kaggle file setup)
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

print(f'train.csv:  {train_df.shape}  |  liquefied rate: {train_df["liquefied"].mean():.1%}')
print(f'test.csv:   {test_df.shape}  |  (no labels)')
print()
print('train.csv head:')
train_df.head()

---

## Part 3 — EDA on Training Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# EDA: TARGET DISTRIBUTION & FEATURE DISTRIBUTIONS BY CLASS
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

features = ['depth', 'N_value', 'fines_content', 'water_table', 'PGA', 'magnitude', 'Vs30']
colors = {'Not Liquefied': '#4C72B0', 'Liquefied': '#C44E52'}

for i, feat in enumerate(features):
    for label_val, label_name in [(0, 'Not Liquefied'), (1, 'Liquefied')]:
        data = train_df[train_df['liquefied'] == label_val][feat]
        axes[i].hist(data, bins=20, alpha=0.6, label=label_name,
                     color=colors[label_name], edgecolor='white')
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel('')
    if i == 0:
        axes[i].legend(fontsize=8)

# Last axis: class balance pie
counts = train_df['liquefied'].value_counts()
axes[7].pie(counts, labels=['Not Liquefied', 'Liquefied'],
            colors=['#4C72B0', '#C44E52'], autopct='%1.1f%%',
            startangle=90)
axes[7].set_title('Class Balance', fontweight='bold')

plt.suptitle('EDA — Feature Distributions by Liquefaction Class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Class distribution:')
print(train_df['liquefied'].value_counts())
print(f'\nClass imbalance ratio: {counts[0]/counts[1]:.2f}:1 (not liquefied : liquefied)')

---

## Part 4 — Feature Engineering for Competition

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING — Competition-grade features
# ─────────────────────────────────────────────────────────────────────────
def engineer_liquefaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add geotechnically motivated features for the liquefaction classifier.

    Features are derived from established geotechnical relationships:
    - Stress normalisation for SPT
    - CSR approximation (cyclic stress ratio)
    - Depth below water table as a proxy for excess pore pressure

    Args:
        df: DataFrame with raw liquefaction features.

    Returns:
        Copy of df with additional engineered columns.
    """
    d = df.copy()

    # Depth below water table (negative = above WT = no liquefaction risk)
    d['depth_below_WT'] = d['depth'] - d['water_table']

    # Overburden stress (simplified, assuming gamma = 18 kN/m³)
    d['sigma_v']         = 18.0 * d['depth']
    d['sigma_v_prime']   = d['sigma_v'] - 9.81 * d['depth_below_WT'].clip(lower=0)

    # Stress normalisation of N value (CN factor, Liao & Whitman 1986)
    d['CN']              = (100.0 / (d['sigma_v_prime'] + 0.01)).clip(upper=2.0) ** 0.5
    d['N1_60']           = (d['N_value'] * d['CN']).clip(upper=60)   # N1,60

    # Cyclic Stress Ratio (Seed & Idriss 1971 — simplified)
    d['rd']              = 1.0 - 0.00765 * d['depth'].clip(upper=9.15)   # stress reduction factor
    d['CSR']             = 0.65 * (d['sigma_v'] / (d['sigma_v_prime'] + 0.01)) * d['PGA'] * d['rd']

    # Magnitude scaling factor (Youd et al. 2001)
    d['MSF']             = 10 ** 2.24 / (d['magnitude'] ** 2.56)
    d['CSR_M']           = d['CSR'] / d['MSF']   # magnitude-scaled CSR

    # Interaction terms
    d['N_PGA']           = d['N_value'] * d['PGA']          # SPT × shaking
    d['fines_N1_60']     = d['fines_content'] * d['N1_60']  # fines effect on resistance
    d['Vs30_bin']        = pd.cut(d['Vs30'], bins=[0, 180, 360, 600],
                                   labels=[0, 1, 2]).astype(float)  # site class

    return d


train_eng = engineer_liquefaction_features(train_df)
test_eng  = engineer_liquefaction_features(test_df)

ENGINEERED = ['depth_below_WT', 'sigma_v_prime', 'N1_60', 'CSR', 'CSR_M', 'MSF',
              'N_PGA', 'fines_N1_60', 'Vs30_bin']

print('New engineered features statistics:')
train_eng[ENGINEERED].describe().round(3)

---

## Part 5 — StratifiedKFold CV with Out-of-Fold (OOF) Predictions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# OOF (OUT-OF-FOLD) PREDICTIONS — CRITICAL FOR RELIABLE CV
# ─────────────────────────────────────────────────────────────────────────
RAW_FEATURES = ['depth', 'N_value', 'fines_content', 'water_table',
                'PGA', 'magnitude', 'Vs30']
ALL_FEATURES = RAW_FEATURES + ENGINEERED
TARGET       = 'liquefied'

X_train = train_eng[ALL_FEATURES].values
y_train = train_eng[TARGET].values
X_test  = test_eng[ALL_FEATURES].values

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Out-of-fold probability arrays
oof_rf  = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
test_rf_folds  = np.zeros((len(X_test), N_FOLDS))
test_xgb_folds = np.zeros((len(X_test), N_FOLDS))

print(f'StratifiedKFold OOF — {N_FOLDS} folds')
print(f'Train size: {len(X_train)} | Test size: {len(X_test)} | Features: {X_train.shape[1]}')
print()
print(f'{"Fold":>5} {"RF AUC":>10} {"XGB AUC":>10}')
print('-' * 28)

scaler = StandardScaler()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # Scale
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    X_te_sc  = scaler.transform(X_test)

    # ── Random Forest ────────────────────────────────────────────────────
    rf = RandomForestClassifier(n_estimators=300, max_depth=8,
                                 min_samples_leaf=2, random_state=42, n_jobs=-1)
    rf.fit(X_tr_sc, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_val_sc)[:, 1]
    test_rf_folds[:, fold] = rf.predict_proba(X_te_sc)[:, 1]
    auc_rf = roc_auc_score(y_val, oof_rf[val_idx])

    # ── XGBoost / GBM fallback ────────────────────────────────────────────
    if HAS_XGB:
        xgb_model = xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, verbosity=0
        )
    else:
        from sklearn.ensemble import GradientBoostingClassifier
        xgb_model = GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
        )

    xgb_model.fit(X_tr_sc, y_tr)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val_sc)[:, 1]
    test_xgb_folds[:, fold] = xgb_model.predict_proba(X_te_sc)[:, 1]
    auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])

    print(f'{fold+1:>5} {auc_rf:>10.4f} {auc_xgb:>10.4f}')

print('-' * 28)
print(f'{"OOF":>5} {roc_auc_score(y_train, oof_rf):>10.4f}'
      f' {roc_auc_score(y_train, oof_xgb):>10.4f}')

---

## Part 6 — Tune RandomForest & XGBoost

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# QUICK HYPERPARAMETER SEARCH (manual grid for transparency)
# ─────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

print('Tuning RandomForestClassifier ...')
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(random_state=42, n_jobs=-1))
])
rf_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth':    [6, 8, None],
    'clf__min_samples_leaf': [1, 2],
}
rf_gs = GridSearchCV(rf_pipe, rf_grid, cv=skf, scoring='roc_auc',
                     n_jobs=-1, refit=True, verbose=0)
rf_gs.fit(X_train, y_train)

print(f'  Best RF params: {rf_gs.best_params_}')
print(f'  Best RF AUC:    {rf_gs.best_score_:.4f}')
print()

if HAS_XGB:
    print('Tuning XGBoostClassifier ...')
    xgb_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    xgb.XGBClassifier(use_label_encoder=False,
                                     eval_metric='logloss',
                                     random_state=42, verbosity=0))
    ])
    xgb_grid = {
        'clf__n_estimators':    [200, 300],
        'clf__max_depth':       [3, 4],
        'clf__learning_rate':   [0.05, 0.10],
        'clf__subsample':       [0.8],
        'clf__colsample_bytree':[0.8],
    }
    xgb_gs = GridSearchCV(xgb_pipe, xgb_grid, cv=skf, scoring='roc_auc',
                          n_jobs=-1, refit=True, verbose=0)
    xgb_gs.fit(X_train, y_train)
    print(f'  Best XGB params: {xgb_gs.best_params_}')
    print(f'  Best XGB AUC:    {xgb_gs.best_score_:.4f}')
else:
    print('Skipping XGBoost tuning (not installed).')
    xgb_gs = rf_gs   # use RF as fallback for blending

---

## Part 7 — Blend Predictions (RF + XGB Weighted Average)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# BLEND: Weighted average of RF and XGB test probabilities
# ─────────────────────────────────────────────────────────────────────────

# Average OOF AUC scores to determine weights
auc_rf_oof  = roc_auc_score(y_train, oof_rf)
auc_xgb_oof = roc_auc_score(y_train, oof_xgb)

w_rf  = auc_rf_oof  / (auc_rf_oof + auc_xgb_oof)
w_xgb = auc_xgb_oof / (auc_rf_oof + auc_xgb_oof)

print(f'OOF AUC — Random Forest: {auc_rf_oof:.4f}  |  XGB/GBM: {auc_xgb_oof:.4f}')
print(f'Blend weights — RF: {w_rf:.3f}  |  XGB: {w_xgb:.3f}')

# Average fold predictions for test set
test_rf_avg  = test_rf_folds.mean(axis=1)
test_xgb_avg = test_xgb_folds.mean(axis=1)

# Blended probabilities
test_blend = w_rf * test_rf_avg + w_xgb * test_xgb_avg

# Blended OOF for validation
oof_blend = w_rf * oof_rf + w_xgb * oof_xgb
auc_blend_oof = roc_auc_score(y_train, oof_blend)

print(f'Blended OOF AUC: {auc_blend_oof:.4f}')
print()

# Visualise OOF probability distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, probs, title in zip(axes,
                             [oof_rf, oof_xgb, oof_blend],
                             ['OOF RF', 'OOF XGB/GBM', 'OOF Blend']):
    for label_val, col in [(0, '#4C72B0'), (1, '#C44E52')]:
        mask = y_train == label_val
        name = 'Not Liquefied' if label_val == 0 else 'Liquefied'
        ax.hist(probs[mask], bins=20, alpha=0.6, color=col, label=name, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted Probability')
    ax.legend(fontsize=8)

plt.suptitle('OOF Probability Distributions by True Class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 8 — Create submission.csv

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CREATE KAGGLE SUBMISSION FILE
# ─────────────────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'id':                       test_df['id'],
    'liquefaction_probability':  test_blend.round(6),
})

submission.to_csv('submission.csv', index=False)

print('submission.csv created.')
print(f'Shape: {submission.shape}')
print()
print('First 10 rows:')
print(submission.head(10).to_string(index=False))
print()
print('Probability statistics:')
print(submission['liquefaction_probability'].describe().round(4))
print()

# Sanity checks
assert len(submission) == len(test_df), 'Row count mismatch!'
assert submission['liquefaction_probability'].between(0, 1).all(), 'Probabilities out of range!'
assert submission['id'].is_unique, 'Duplicate IDs in submission!'
print('All sanity checks passed.')

---

## Part 9 — Competition Tips for Geotechnical ML

### Feature Engineering Ideas

| Idea | Geotechnical Rationale |
|------|------------------------|
| `N1_60 = N60 × CN` | Stress-normalised SPT is more physically meaningful than raw N |
| `CSR = 0.65 × (σv/σ'v) × PGA × rd` | Direct measure of earthquake-induced shear stress |
| `depth_below_WT` | Only layers below water table can liquefy |
| `MSF = 10^2.24 / Mw^2.56` | Magnitude scaling reduces CSR for smaller earthquakes |
| `Vs30_bin` | Site class categorisation (NEHRP A–E) captures non-linearity |
| Log of Vs30 | Log-linear relationship with amplification |
| `depth × PGA` | Deeper layers with high PGA = higher risk |

### Avoiding Data Leakage

```
LEAKAGE SOURCES IN COMPETITION DATA:

1. Fitting scalers/encoders on ALL data (train + test)
   Fix: fit only on train fold; transform test fold separately

2. Using post-earthquake observations in features
   (e.g., surface cracking, sand boils observed after the event)
   Fix: use only pre-earthquake site investigation data

3. Using the same feature-generation logic as the label
   (e.g., if CSR/CRR > 1.0 is the label, CSR is a leaky feature)
   Fix: use CSR as a feature but not the exact trigger condition

4. Target-mean encoding without proper CV isolation
   Fix: always compute target statistics inside CV folds
```

### Metric Choice: Why AUC-ROC for Liquefaction?

```
Class balance: ~25% liquefied, ~75% not liquefied (imbalanced)

Accuracy is misleading: a model predicting ALL 'not liquefied'
would score 75% accuracy but catch 0% of actual liquefaction events.

AUC-ROC measures:
  - Probability that a randomly chosen liquefied site ranks
    higher than a randomly chosen non-liquefied site
  - Insensitive to class imbalance
  - 0.5 = random | 1.0 = perfect | 0.9+ = excellent

For life-safety problems, also consider:
  - Precision-Recall AUC (when false negatives are catastrophic)
  - F1 at specific thresholds (when you must classify, not rank)
```

---

## Key Takeaways

1. **OOF (out-of-fold) predictions are the gold standard for CV in competitions.** They give a single, unbiased probability estimate for every training sample — comparable to the test set predictions — and a more reliable AUC than simple cross-validation.

2. **Geotechnically derived features beat raw features.** CSR, N1,60, depth_below_WT, and MSF encode decades of empirical research. Domain knowledge is your biggest competitive advantage when data is limited.

3. **Blending diverse models reduces variance.** RF and XGBoost make different errors. A weighted average (weights proportional to OOF AUC) consistently outperforms either model alone — often by 0.005–0.020 AUC points.

4. **AUC-ROC is the right metric for imbalanced safety-critical problems.** Accuracy would be 75% even for a model that never predicts liquefaction. AUC rewards correctly ranking high-risk sites above low-risk ones.

5. **Validate your submission file before uploading.** Check: row count matches test set, probabilities in [0,1], no duplicate IDs, column names exactly match the sample submission format.

---

*Day 099 of 100 — One notebook away from the finish line. From borehole data to Kaggle leaderboard!*